# **1. PLM**

PLM(Pre-trained Language Model)은 **대량의 텍스트 데이터를 사전 학습하여 자연어 이해와 생성 능력을 갖춘 인공지능 모델**이다. 대표적으로 BERT, GPT, T5 등이 있으며, 이들은 대규모 데이터에서 단어의 의미와 문맥을 학습한 후, 특정 작업(예: 문장 분류, 번역, 질의응답 등)에 맞게 추가 학습(Fine-tuning)하여 활용된다. PLM은 문맥을 고려한 자연어 처리 능력이 뛰어나며, 다양한 언어 기반 AI 애플리케이션에서 핵심 기술로 사용된다.

In [5]:
import pandas as pd
import numpy as np
import random
import time
import datetime
import csv
import os
import torch
import torch.nn.functional as F

from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import load_dataset
from torch.optim import AdamW
from transformers import pipeline
from transformers import BertTokenizer
from transformers import BertForSequenceClassification, BertConfig
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, hamming_loss

c:\MCP\agentlog\mcp04_multimodal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **2. K-MHaS 데이터셋**

In [1]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("./data")
REPO_DIR = PROJECT_DIR / 'K-MHaS'
DATA_DIR = REPO_DIR / 'data'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/adlnlp/K-MHaS.git', str(REPO_DIR)], check=True)
else:
    print(f'이미 저장소가 존재합니다: {REPO_DIR}')

print('K-MHaS 폴더:')
for p in REPO_DIR.iterdir():
    print(' -', p.name)

print('\ndata 폴더:')
for p in DATA_DIR.iterdir():
    print(' -', p.name)

K-MHaS 폴더:
 - .git
 - data
 - README.md
 - resource

data 폴더:
 - kmhas_test.txt
 - kmhas_train.txt
 - kmhas_valid.txt


In [2]:
train_file = DATA_DIR / 'kmhas_train.txt'

with train_file.open('r', encoding='utf-8') as f:
    for _ in range(3):
        print(f.readline().rstrip())

document	label
"자한당틀딱들.. 악플질 고만해라."	2,4
정치적으로 편향된 평론한은 분은 별로...	8


In [6]:
# split='train'은 실제 train 의미라기 보다는 Hugging Face가 생성한 단일 split를 바로 꺼내겠다는 의미
train = load_dataset('csv', data_files=str(DATA_DIR / 'kmhas_train.txt'), delimiter='\t', split='train')
validation = load_dataset('csv', data_files=str(DATA_DIR / 'kmhas_valid.txt'), delimiter='\t', split='train')
test = load_dataset('csv', data_files=str(DATA_DIR / 'kmhas_test.txt'), delimiter='\t', split='train')

print("[train]\n", train)
print("[val]\n", validation)
print("[test]\n", test)

Generating train split: 78977 examples [00:00, 136145.38 examples/s]
Generating train split: 8776 examples [00:00, 140532.33 examples/s]
Generating train split: 21939 examples [00:00, 173115.80 examples/s]

[train]
 Dataset({
    features: ['document', 'label'],
    num_rows: 78977
})
[val]
 Dataset({
    features: ['document', 'label'],
    num_rows: 8776
})
[test]
 Dataset({
    features: ['document', 'label'],
    num_rows: 21939
})


In [ ]:
train[0]

In [ ]:
train[0]['document']

In [ ]:
train[0]['label']

### 특수 토큰 추가
- [CLS]: 문장 시작에 배치되는 분류용 토큰
- [SEP]: 문장의 끝 또는 문장 사이를 구분하는 토큰

In [ ]:
train_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', train['document']))
validation_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', validation['document']))
test_sentences = list(map(lambda x: '[CLS] ' + str(x) + ' [SEP]', test['document']))

In [ ]:
train_sentences[:5]

In [ ]:
def parse_label(s):
    return [int(x) for x in str(s).split(',')]

In [ ]:
train_y = [parse_label(s) for s in train['label']]
validation_y = [parse_label(s) for s in validation['label']]
test_y = [parse_label(s) for s in test['label']]

In [ ]:
train_y[:5]

신경망의 출력층은 클래스마다 하나의 값을 출력해야 한다.
- 예시) [2, 3] 같은 가변 길이 label을 고정 길이 벡터로 바꿔야 한다.
```
클래스가 0 ~ 4이고 정답이 [1, 3]이라면 [0, 1, 0, 1, 0] # Multi-Hot encoding
```

In [ ]:
# 학습 데이터에 등장하는 클래스의 종류를 파악
# num_labels: 출력 클래스 개수
enc = MultiLabelBinarizer()
enc.fit(train_y)
num_labels = len(enc.classes_)
num_labels, enc.classes_

In [ ]:
train_labels = torch.tensor(enc.transform(train_y), dtype=torch.float32)
validation_labels = torch.tensor(enc.transform(validation_y), dtype=torch.float32)
test_labels = torch.tensor(enc.transform(test_y), dtype=torch.float32)

In [ ]:
train_labels[:5]

In [ ]:
test[:5]

In [ ]:
test_sentences[:5]

In [ ]:
test_labels[:5]

Trian/Validation/Test 각각 별도의 MultiLabelBinarizer를 만들지 않고, 클래스 인덱스의 의미가 일치하도록 하기 위해 train에서 fit한 동일 enc를 사용했다.

In [ ]:
# KLUE-BERT가 맞는 vocabulary와 tokenizer 설정을 내려받아 로드
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')

In [ ]:
# 문장 길이는 서로 다르지만 mini-batch는 동일한 크기의 tensor여야 하므로, 짧은 문장에는 [PAD]를 붙이고, 긴 문장은 MAX_LEN=128로 자름
MAX_LEN = 128

def data_to_tensor(sentences, labels, tokenizer, max_len=MAX_LEN):
    pad_id = tokenizer.pad_token_id
    input_ids = []

    for sent in sentences:
        sent = "" if sent is None else str(sent)  # 결측치(None/NaN)가 섞여 있어도 에러 없이 빈 문자열로 처리
        # BERT류 모델은 문장 맨 앞 [CLS], 맨 뒤 [SEP]가 있어야 하므로, 없으면 붙여줌
        if not sent.startswith("[CLS]"):
            sent = '[CLS] ' + sent
        if not sent.endswith("[SEP]"):
            sent = sent + ' [SEP]'

        tokens = tokenizer.tokenize(sent)

        # max_len을 넘으면 잘라내되, 잘린 뒤에도 문장 끝은 [SEP]로 끝나야 하므로 마지막 토큰을 [SEP]로 교체
        if len(tokens) > max_len:
            tokens = tokens[:max_len]
            if tokens[-1] != '[SEP]':
                tokens[-1] = '[SEP]'

        ids = tokenizer.convert_tokens_to_ids(tokens)
        input_ids.append(torch.tensor(ids, dtype=torch.long))

    padded_inputs = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)

    # 배치 결과 길이를 항상 MAX_LEN으로 고정: 모자라면 뒤에 [PAD] 추가, 넘치면 잘라냄
    if padded_inputs.size(1) < max_len:
        padded_inputs = F.pad(padded_inputs, (0, max_len - padded_inputs.size(1)), value=pad_id)
    else:
        padded_inputs = padded_inputs[:, :max_len]

    attention_masks = (padded_inputs != pad_id).float()  # 실제 토큰은 1, [PAD] 자리는 0으로 표시해 attention에서 PAD를 무시하게 함

    # labels가 이미 텐서로 들어올 수도, 리스트/배열로 들어올 수도 있어서 둘 다 대응
    if torch.is_tensor(labels):
        tensor_labels = labels.to(dtype=torch.float32)
    else:
        tensor_labels = torch.tensor(labels, dtype=torch.float32)

    return padded_inputs, tensor_labels, attention_masks

In [ ]:
# *_inputs: Token ID tensor
# *_labels: Multi-Hot 정답 tensor
# *_mask: Attention  mask tensor
train_inputs, train_labels, train_mask = data_to_tensor(train_sentences, train_labels, tokenizer)
validation_inputs, validation_labels, validation_mask = data_to_tensor(validation_sentences, validation_labels, tokenizer)
test_inputs, test_labels, test_mask = data_to_tensor(test_sentences, test_labels, tokenizer)

In [ ]:
train_inputs, train_labels, train_mask

In [ ]:
batch_size = 32

train_data = TensorDataset(train_inputs, train_mask, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

validation_data = TensorDataset(validation_inputs, validation_mask, validation_labels)
validation_sampler = RandomSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

test_data = TensorDataset(test_inputs, test_mask, test_labels)
test_sampler = RandomSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

len(train_labels), len(validation_labels), len(test_labels)

# **3. KLUE-BERT 다중 레이블 분류 모델**

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

In [ ]:
num_labels = len(enc.classes_)
model = BertForSequenceClassification.from_pretrained(
    'klue/bert-base',
    num_labels=num_labels,
    problem_type='multi_label_classification'
)
model.to(DEVICE)

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 10
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
# time.time()으로 잰 두 시점의 차이(초 단위 숫자)를 시:분:초 형태의 문자열로 바꿔주는 함수
def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
def multi_label_metrics(predictions, labels, threshold=0.5):
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    y_true = labels

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0)
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0)
    f1_weighted_average = f1_score(y_true=y_true, y_pred=y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_true, y_pred, average='micro')

    metrics = {
        'accuracy': accuracy,
        'f1_macro_average': f1_macro_average,
        'f1_micro_average': f1_micro_average,
        'f1_weighted_average': f1_weighted_average,
        'roc_auc': roc_auc
    }
    return metrics

In [ ]:
seed_val = 2026
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_val)

In [ ]:
model.zero_grad()

for epoch_i in range(0, epochs):
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    t0 = time.time()
    total_loss = 0

    model.train()

    for step, batch in tqdm(enumerate(train_dataloader)):
        if step % 500 == 0 and not step == 0:
            elapsed = format_time(time.time() - t0)
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        batch = tuple(t.to(DEVICE) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
        loss = outputs[0]
        total_loss += loss.item()
        loss.backward()
        # L2 norm이 1.0을 넘으면 강제로 잘라냄
        # BERT 학습에서 필수 코드
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        model.zero_grad()
    avg_train_loss = total_loss / len(train_dataloader)
    print("")
    print("  Average training loss: {0:.4f}".format(avg_train_loss))
    print("  Training epoch took: {:}".format(format_time(time.time() - t0)))

In [ ]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for batch in validation_dataloader:
    batch = tuple(t.to(DEVICE) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro_average']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro_average']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted_average']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))

In [ ]:
from pathlib import Path

MODEL_DIR = Path.cwd() / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('모델 저장 폴더:', MODEL_DIR.resolve())

In [ ]:
model_path = MODEL_DIR / 'BERT_multilabel_model.pt'
torch.save(model.state_dict(), model_path)
print('저장 완료:', model_path.resolve())

In [ ]:
state_dict = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
print('모델 로드 완료')

In [ ]:
t0 = time.time()
model.eval()
accum_logits, accum_label_ids = [], []

for step, batch in tqdm(enumerate(test_dataloader)):
    if step % 100 == 0 and not step == 0:
        elapsed = format_time(time.time() - t0)
        print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(test_dataloader), elapsed))

    batch = tuple(t.to(DEVICE) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    for b in logits:
        accum_logits.append(list(b))

    for b in label_ids:
        accum_label_ids.append(list(b))

accum_logits = np.array(accum_logits)
accum_label_ids = np.array(accum_label_ids)
results = multi_label_metrics(accum_logits, accum_label_ids)

print("Accuracy: {0:.4f}".format(results['accuracy']))
print("F1 (Macro) Score: {0:.4f}".format(results['f1_macro_average']))
print("F1 (Micro) Score: {0:.4f}".format(results['f1_micro_average']))
print("F1 (Weighted) Score: {0:.4f}".format(results['f1_weighted_average']))
print("ROC-AUC: {0:.4f}".format(results['roc_auc']))

# **4. Hugging Face pipeline으로 추론 인터페이스 만들기**

Pipeline()
```
Tokenizer > Tensor 변환 > Model 추론 > 결과 후 처리
```
하나로 묶어주는 Hugging Face의 API이다.

In [ ]:
pipe_device = 0 if torch.cuda.is_available() else -1

# GPU 사용 device=0, CPU 사용 device=-1
# truncation=True: 너무 긴 문장을 잘라줌
# function_to_apply='sigmoid': logits를 각 클래스의 0 ~ 1 score로 변환
# top_k=None: 특정 상위 몇 개만 자르지 말고 모든 label 결과를 반환
pipe = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=pipe_device,
    truncation=True,
    max_length=512,
    function_to_apply='sigmoid',
    top_k=None
)

In [ ]:
result = pipe(" ")  # " "에 실제 문장을 넣어 결과를 확인할 수 있음
print(result)

In [ ]:
label_dict = {'LABEL_0' : '출신차별', 'LABEL_1' : '외모차별', 'LABEL_2' : '정치성향차별', \
              'LABEL_3': '혐오욕설', 'LABEL_4': '연령차별', 'LABEL_5': '성차별', 'LABEL_6' : '인종차별', \
              'LABEL_7': '종교차별', 'LABEL_8': '해당사항없음'}

In [ ]:
def prediction(text):
    result = pipe(text)
    return [label_dict[res['label']] for res in result[0] if res['score'] > 0.5]

In [ ]:
prediction(" ")